1.

In [44]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cross_decomposition import PLSRegression
from google.colab import files
import warnings

# Suppress warnings for clean output
warnings.filterwarnings('ignore')


2.

In [45]:
def calculate_vips(model):
    """Calculate VIP (Variable Importance in Projection) scores for a PLS model."""
    t = model.x_scores_
    w = model.x_weights_
    q = model.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    for i in range(p):
        weight = np.array([ (w[i,j] / np.linalg.norm(w[:,j]))**2 for j in range(h) ])
        vips[i] = np.sqrt(p * (s.T @ weight)/total_s)
    return vips

3.

In [46]:
def full_pipeline_exact_output():
    # 1. Load the datasets using direct-download Google Drive URLs
    class_id = "1sTD_7AAc0Qrj9suia_n6Sj6uhKUJGn3g"
    pred_id = "18V3JyPKJDsSdjgk9aYgh-h6wmCl8kB7V"

    class_url = f"https://drive.google.com/uc?export=download&id={class_id}"
    pred_url = f"https://drive.google.com/uc?export=download&id={pred_id}"

    print("Downloading datasets from Google Drive...\n")
    df_class_raw = pd.read_csv(class_url)
    df_pred_raw = pd.read_csv(pred_url)

4.

In [47]:
    # Identifiers to drop immediately (We DO NOT drop Rail here, so the math matches your exact output)
    identifiers = ['SI', 'Asset Name', 'NBI 007: Facility Carried by Structure', 'NBI 006: Feature Intersected']

5.

In [48]:
    # =====================================================================
    # PART A: FEATURE SELECTION (Mathematical Proof for your Report)
    # =====================================================================
    for df_raw, task in zip([df_class_raw, df_pred_raw], ['Classification', 'Prediction']):
        print(f"{'='*70}")
        print(f"--- Feature Selection for {task} Task ---")
        print(f"{'='*70}")

        df_temp = df_raw.copy()
        df_temp = df_temp.drop(columns=[col for col in identifiers if col in df_temp.columns])

        target_col = 'Foundation Type' if task == 'Classification' else 'Pier total Depth (ft)'

        # Handle "U" / "Unknown" Artifacts
        df_temp.replace(['U', 'Unknown', 'Unknown ', ' '], np.nan, inplace=True)

        # Feature Engineering: Bridge Age = 2025 - Year Built
        if 'Years Build' in df_temp.columns:
            df_temp['Years Build'] = pd.to_numeric(df_temp['Years Build'], errors='coerce')
            df_temp['Bridge Age'] = 2025 - df_temp['Years Build']
            df_temp.drop(columns=['Years Build'], inplace=True)

        # Encoding Target Variable (Classification Only)
        if task == 'Classification':
            df_temp[target_col] = df_temp[target_col].map({'Deep': 1, 'Shallow': 0})
            df_temp.dropna(subset=[target_col], inplace=True)

        X = df_temp.drop(columns=[target_col])
        y = df_temp[target_col]

        num_cols = X.select_dtypes(include=['int64', 'float64', 'Int64']).columns
        cat_cols = X.select_dtypes(include=['object', 'category']).columns

        # Imputation
        for col in num_cols: X[col].fillna(X[col].median(), inplace=True)
        for col in cat_cols: X[col].fillna(X[col].mode()[0], inplace=True)

        # Encoding
        X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

        # Scaling
        scaler = StandardScaler()
        X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

        # 1. Correlation
        df_corr = X_scaled.copy()
        df_corr['TARGET'] = y.values
        correlations = df_corr.corr()['TARGET'].drop('TARGET').abs().sort_values(ascending=False)
        print("\nTop 10 Features by Absolute Correlation (Linear Relationship):")
        print(correlations.head(10))

        # 2. Random Forest
        model_rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=3) if task == 'Classification' else RandomForestRegressor(n_estimators=100, random_state=42, max_depth=3)
        model_rf.fit(X_scaled, y)
        importances = pd.Series(model_rf.feature_importances_, index=X_scaled.columns).sort_values(ascending=False)
        print("\nTop 10 Features by Random Forest Importance (Non-Linear Relationship):")
        print(importances.head(10))

        # 3. PLS-VIP
        model_pls = PLSRegression(n_components=2)
        model_pls.fit(X_scaled, y)
        vip_scores = pd.Series(calculate_vips(model_pls).flatten(), index=X_scaled.columns).sort_values(ascending=False)
        print("\nTop 10 Features by PLS-VIP Score (Scores > 1.0 indicate high importance):")
        print(vip_scores.head(10))
        print("\n\n")

--- Feature Selection for Classification Task ---

Top 10 Features by Absolute Correlation (Linear Relationship):
Pier Shape_Square        0.804762
Bridge Age               0.616859
Girder Type_T-beam       0.581374
Rail Type_NS             0.508549
Max Span Length (ft)     0.505225
Min Span Length (ft)     0.449117
Pier Shape_Wall          0.443658
Rail Height (in)_NS      0.359611
Rail Height (in)_34      0.359611
Pier size/dia (in)_27    0.359611
Name: TARGET, dtype: float64

Top 10 Features by Random Forest Importance (Non-Linear Relationship):
Bridge Age              0.123986
Roadway Width (ft)      0.103190
Pier Shape_Square       0.100892
Min Span Length (ft)    0.089191
Max Span Length (ft)    0.084876
Deck Width (ft)         0.058457
Girder Type_T-beam      0.048866
Rail Type_NS            0.032525
Total Length (ft)       0.030781
Pier Shape_Wall         0.029581
dtype: float64

Top 10 Features by PLS-VIP Score (Scores > 1.0 indicate high importance):
Pier Shape_Square        

6.

In [49]:

    # =====================================================================
    # PART B: GENERATE FINAL PROCESSED DATASETS (Applying Engineering Judgment)
    # =====================================================================
    print(f"{'='*70}")
    print("--- Generating and Downloading Final Processed Datasets ---")
    print(f"{'='*70}")

    # Process Classification Data
    df_class = df_class_raw.copy()
    df_class.replace(['U', 'Unknown', 'Unknown ', ' '], np.nan, inplace=True)
    df_class['Years Build'] = pd.to_numeric(df_class['Years Build'], errors='coerce')
    df_class['Bridge Age'] = 2025 - df_class['Years Build']

    # Selected 7 Predictors (Rail Type/Height are explicitly excluded here!)
    class_features = [
        'Bridge Age', 'Pier Shape', 'Max Span Length (ft)', 'Girder Type',
        'Roadway Width (ft)', 'Total Length (ft)', 'Pier size/dia (in)', 'Foundation Type'
    ]
    df_class_processed = df_class[class_features]
    class_filename = "processed_classification_dataset.csv"
    df_class_processed.to_csv(class_filename, index=False)

    # Process Prediction Data
    df_pred = df_pred_raw.copy()
    df_pred.replace(['U', 'Unknown', 'Unknown ', ' '], np.nan, inplace=True)
    df_pred['Years Build'] = pd.to_numeric(df_pred['Years Build'], errors='coerce')
    df_pred['Bridge Age'] = 2025 - df_pred['Years Build']

    # Selected 7 Predictors (Rail Type/Height are explicitly excluded here!)
    pred_features = [
        'Pier unbraced length (ft)', 'Max Span Length (ft)', 'Total Length (ft)',
        'Pier size/dia (in)', 'Avg Daily Trafic', 'Regulatory Speed (mph)', 'Bridge Age', 'Pier total Depth (ft)'
    ]
    df_pred_processed = df_pred[pred_features]
    pred_filename = "processed_prediction_dataset.csv"
    df_pred_processed.to_csv(pred_filename, index=False)

    print(f"Created '{class_filename}' with shape {df_class_processed.shape}")
    print(f"Created '{pred_filename}' with shape {df_pred_processed.shape}")


--- Generating and Downloading Final Processed Datasets ---
Created 'processed_classification_dataset.csv' with shape (71, 8)
Created 'processed_prediction_dataset.csv' with shape (71, 8)


7.

In [50]:
    # Trigger downloads in Colab
    print("Initiating downloads...")
    files.download(class_filename)
    files.download(pred_filename)

# Execute
full_pipeline_exact_output()

Initiating downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>